# ViFinQA — BGE-M3 Synthetic Retriever V1

Fine-tune retriever từ curriculum `synthetic_execution_verified`. Notebook chỉ dùng report-table inventory độc lập và không đọc benchmark questions.

**Điều kiện đạt:** tất cả manifest/hash/replay gates pass, model artifact và `training_metadata.json` được ghi vào `/kaggle/working/bge_m3_synthetic_v1/`. Artifact này chưa được promote cho submission; cần đánh giá issuer-held-out sau training.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import shutil

REPO_SLUG = 'Dle28/nlp-finance-query-'
REPO_REF = 'codex/synthetic-finance-curriculum-v1'
REPO_DIR = Path('/kaggle/working/AI_guru')
SECRET_NAMES = ('GIT_TOKEN', 'GITHUB_TOKEN')
SOURCE_TREE_NAME = 'ai_guru_synthetic_retriever_source_v1'
SOURCE_MANIFEST_NAME = 'ai_guru_synthetic_retriever_source_v1.manifest.json'

source_dirs = sorted({
    path.parent for path in Path('/kaggle/input').rglob(SOURCE_MANIFEST_NAME)
    if (path.parent / SOURCE_TREE_NAME).is_dir()
})
if len(source_dirs) > 1:
    raise RuntimeError(f'Expected at most one synthetic source snapshot; found {len(source_dirs)}.')

token = None
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            token = client.get_secret(name)
        except Exception:
            continue
        if token:
            break
except Exception:
    pass

git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
basic = None
try:
    if token:
        basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        git_env.update({
            'GIT_CONFIG_COUNT': '1',
            'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {basic}',
        })
        if REPO_DIR.exists():
            subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_REF], env=git_env, check=True)
            subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], env=git_env, check=True)
        else:
            subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, f'https://github.com/{REPO_SLUG}.git', str(REPO_DIR)], env=git_env, check=True)
        revision = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
        source_mode = 'github_secret'
    elif source_dirs:
        source_dir = source_dirs[0]
        manifest = json.loads((source_dir / SOURCE_MANIFEST_NAME).read_text(encoding='utf-8'))
        expected_tree_sha = manifest.get('source_tree_sha256')
        expected_file_count = manifest.get('source_tree_file_count')
        if manifest.get('protocol') != 'kaggle_synthetic_retriever_source_v1' or not isinstance(expected_tree_sha, str) or not isinstance(expected_file_count, int):
            raise ValueError('Synthetic source manifest is missing the verified source-tree contract.')
        source_root = source_dir / SOURCE_TREE_NAME
        actual_files = {
            path.relative_to(source_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
            for path in source_root.rglob('*') if path.is_file()
        }
        actual_tree_sha = hashlib.sha256(json.dumps(actual_files, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        if actual_tree_sha != expected_tree_sha or len(actual_files) != expected_file_count:
            raise ValueError('Synthetic source tree failed manifest/hash verification.')
        if any(path == 'data/ViFinQA' or path.startswith('data/ViFinQA/') for path in actual_files):
            raise ValueError('Synthetic source snapshot must not contain benchmark data.')
        if REPO_DIR.exists():
            raise RuntimeError(f'Refusing to merge source snapshot into existing path: {REPO_DIR}')
        shutil.copytree(source_root, REPO_DIR)
        if not (REPO_DIR / 'scripts' / 'train_synthetic_retriever_v1.py').is_file():
            raise FileNotFoundError('Synthetic source snapshot is missing the trainer.')
        revision = str(manifest.get('git_commit') or 'source-snapshot')
        source_mode = 'tree_hash_verified_source_snapshot'
    else:
        raise RuntimeError('Attach the synthetic source snapshot or add GIT_TOKEN/GITHUB_TOKEN as a Kaggle secret.')
finally:
    token = None
    basic = None
    git_env = None

print({'repo': str(REPO_DIR), 'revision': revision, 'ref': REPO_REF, 'source_mode': source_mode})


In [ ]:
import json

# Pin a Pascal-compatible CUDA stack before importing sentence-transformers.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch==2.12.1', 'torchvision==0.27.1',
    '--index-url', 'https://download.pytorch.org/whl/cu126',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'sentence-transformers==3.4.1', 'transformers==4.48.3',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, restart, then Run All.')
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gib < 14:
    raise RuntimeError(f'BGE-M3 synthetic fine-tune requires >=14 GiB VRAM; found {vram_gib:.1f} GiB.')
print({'gpu': torch.cuda.get_device_name(0), 'vram_gib': round(vram_gib, 2), 'torch': torch.__version__})

## Inputs and preflight gates

Attach exactly these private datasets in the Kaggle Input panel:

- `dungle2810/vifinqa-synthetic-finance-curriculum-v1`
- `dungle2810/vifinqa-baseline-artifacts`

The next cell requires exactly one curriculum/manifest pair and exactly one independent `table_assets.jsonl`, then verifies their SHA-256 relationship before training.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
CURRICULUM_NAME = 'synthetic_finance_curriculum_v1.jsonl'
MANIFEST_NAME = 'synthetic_finance_curriculum_v1.manifest.json'

curriculum_dirs = sorted({
    candidate.parent for candidate in INPUT_ROOT.rglob(CURRICULUM_NAME)
    if (candidate.parent / MANIFEST_NAME).is_file()
})
if len(curriculum_dirs) != 1:
    raise RuntimeError(f'Expected exactly one curriculum input directory; found {len(curriculum_dirs)}.')
CURRICULUM_DIR = curriculum_dirs[0]
CURRICULUM = CURRICULUM_DIR / CURRICULUM_NAME
MANIFEST = CURRICULUM_DIR / MANIFEST_NAME

table_assets = sorted(INPUT_ROOT.rglob('table_assets.jsonl'))
if len(table_assets) != 1:
    raise RuntimeError(f'Expected exactly one baseline table_assets.jsonl input; found {len(table_assets)}.')
TABLES = table_assets[0]

trainer = REPO_DIR / 'scripts' / 'train_synthetic_retriever_v1.py'
if not trainer.is_file():
    raise FileNotFoundError(f'Missing synthetic trainer in checked-out revision: {trainer}')

preflight = subprocess.run([
    sys.executable, '-c',
    'import importlib.util, pathlib; '
    'p=pathlib.Path(\"scripts/train_synthetic_retriever_v1.py\"); '
    's=importlib.util.spec_from_file_location(\"trainer\", p); m=importlib.util.module_from_spec(s); s.loader.exec_module(m); '
    f'print(m.validate_manifest(pathlib.Path({str(CURRICULUM)!r}), pathlib.Path({str(MANIFEST)!r}), pathlib.Path({str(TABLES)!r}))[\"summary\"])'
], cwd=REPO_DIR, capture_output=True, text=True)
if preflight.returncode:
    raise RuntimeError(preflight.stderr[-4000:])
print({'curriculum': str(CURRICULUM), 'tables': str(TABLES), 'manifest_summary': preflight.stdout.strip()})

## Train

Batch size is intentionally 2 for a 16 GiB Kaggle P100/T4-compatible run. The trainer uses each explicit hard negative plus in-batch negatives. It does not read benchmark questions, produce a submission, or promote provenance.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/bge_m3_synthetic_v1')
command = [
    sys.executable, str(trainer),
    '--curriculum', str(CURRICULUM),
    '--manifest', str(MANIFEST),
    '--bundle-tables', str(TABLES),
    '--output-dir', str(OUTPUT_DIR),
    '--split', 'train', '--model', 'BAAI/bge-m3',
    '--epochs', '3', '--batch-size', '2', '--max-seq-length', '384',
    '--device', 'cuda:0', '--gpu-id', '0', '--gradient-checkpointing',
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)
metadata = json.loads((OUTPUT_DIR / 'training_metadata.json').read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))

## Result contract

Download the complete `/kaggle/working/bge_m3_synthetic_v1/` directory. Before replacing any retriever, evaluate this model only on the issuer-held-out validation/test splits and compare Recall@K plus wrong-scope/wrong-year rates. Do not treat successful training as evidence correctness or submission eligibility.

In [ ]:
required = ['training_metadata.json', 'config_sentence_transformers.json']
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Training output is incomplete: {missing}')
print({'output_dir': str(OUTPUT_DIR), 'files': sorted(path.name for path in OUTPUT_DIR.iterdir()), 'status': 'training_artifact_ready_for_offline_evaluation'})